# Capital One — Data Science Assessment (Simulation, Round 2)

Same four-question shape as round 1, different domain (**food-delivery couriers**) and deliberately
different formats: JSON and CSV inputs, a JSON output for Q1, an extra lookup table to join, and a
non-ISO date format.

Suggested budget: **~70 minutes** (Q1 10m, Q2 20m, Q3 20m, Q4 20m). Try it closed-book this time.

Regenerate fresh data with `python generate_data.py`.

In [2]:
%load_ext autoreload
%autoreload 2

import json, os, time
import numpy as np
import pandas as pd

import grader

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("ready")

ready


---
## Question 1 of 4 — Basic data analysis

You are provided with data about food-delivery couriers and their orders. Perform basic analysis and save
the results to a JSON file. Treat **June 1st, 2024 as today**.

Data (in `q1/`):

**`couriers.json`** — a JSON array of records:
- `courier_id` (int), `age` (int)
- `joined_date` (str, `YYYY-MM-DD`) — the day the courier joined the platform
- `rating` (float)
- `vehicle_type` (str) — one of `["Bicycle", "Car", "E-bike", "Scooter", "Walking"]`

**`orders_{i}.csv`** — split into 3 files, `orders_1.csv` … `orders_3.csv`:
- `order_id` (int), `courier_id` (int), `customer_id` (int), `order_date` (str)
- `status` (str) — one of `["Delivered", "Cancelled by customer", "Unassigned"]`
- `order_value` (float)

**Tasks**
1. **`average_courier_tenure_days`** — mean number of days between each courier's `joined_date` and today.
2. **`percentage_couriers_with_ebike`** — percentage of couriers whose `vehicle_type` is `"E-bike"`.
3. **`median_order_value`** — median `order_value` across **all** orders (combine all three files).
4. **`order_cancellation_rate`** — percentage of all orders with `status == "Cancelled by customer"`.

**Output:** save `q1/analysis_results.json` as a single JSON object mapping each of the four names above to
its numeric value, e.g.

```json
{"average_courier_tenure_days": 0.0, "percentage_couriers_with_ebike": 0.0,
 "median_order_value": 0.0, "order_cancellation_rate": 0.0}
```

Values are correct if they match to two decimal places. `q1/tests/expected_format.json` shows the expected
format and the true value of `average_courier_tenure_days`; the other three are zero placeholders.

In [40]:
# YOUR SOLUTION — Question 1
_t0 = time.time()

couriers = pd.read_json("q1/couriers.json")
orders = pd.concat([pd.read_csv(f"q1/orders_{i}.csv") for i in range(1, 4)], ignore_index=True)
TODAY = pd.Timestamp("2024-06-01")

couriers['total_days']=(
    (TODAY - pd.to_datetime(couriers['joined_date'])).dt.days
)
average_courier_days = couriers['total_days'].mean()
print(average_courier_days)
num_with_ebike = len(couriers[
    couriers['vehicle_type']=="E-bike"
])
percentage_couriers_with_ebike = (num_with_ebike/len(couriers))*100

median_order_value = orders['order_value'].median()
# print(median_order_value)
# print(percentage_couriers_with_ebike)
# print(couriers)

num_cancels = len(orders[
    orders['status']=="Cancelled by customer"
])

order_cancellation_rate = (num_cancels/len(orders))*100
print(order_cancellation_rate)
# ... write q1/analysis_results.json

output = pd.Series({
    "average_courier_tenure_days": average_courier_days,
    "percentage_couriers_with_ebike": percentage_couriers_with_ebike,
    "median_order_value": median_order_value,
    "order_cancellation_rate": order_cancellation_rate
})

output.to_json("q1/analysis_results.json")
# print(f"runtime: {time.time() - _t0:.2f}s")

901.8108
11.06


In [41]:
grader.grade_q1()

=== Round 2 / Question 1: basic analysis ===
[PASS] analysis_results.json exists  -> /Users/log/Github/CPT/capital1_oa/round2/q1/analysis_results.json
[PASS] top level is a JSON object  -> dict
[PASS] no unexpected keys  -> extra []
[PASS] average_courier_tenure_days == 901.81  -> got 901.8108
[PASS] percentage_couriers_with_ebike == 24.52  -> got 24.5200
[PASS] median_order_value == 27.57  -> got 27.5700
[PASS] order_cancellation_rate == 11.06  -> got 11.0600
--- 7/7 checks passed ---


True

---
## Question 2 of 4 — Feature collection

Data about couriers and their orders, created by **June 1st, 2024**. When calculating any time features,
treat **June 1st, 2024 as today**.

Data (in `q2/`), across 6 files:

**`couriers.json`** (JSON array) — `courier_id` (int), `vehicle_id` (int), `zone_id` (int), `age` (int),
`joined_date` (str, `YYYY-MM-DD`), `rating` (float), `total_earnings` (float),
`courier_tier` (str: `"Gold"` / `"Standard"`)

**`vehicles.csv`** — `vehicle_id` (int), `vehicle_type` (str), `purchase_year` (int),
`last_service_date` (str — ⚠️ formatted **`DD/MM/YYYY`**)

**`zones.csv`** — `zone_id` (int), `city` (str), `region` (str), `zone_name` (str)

**`orders_{i}.csv`** (3 files) — `order_id`, `courier_id`, `customer_id`, `order_date`,
`status`, `order_value`, `delivery_minutes` (float, empty unless delivered), `rating_given`
(float 1–5, empty unless delivered), `on_time` (bool), `complaint_filed` (bool)

**Task:** collect per-courier information into **`q2/collected.csv`** with columns:

| column | type | notes |
|---|---|---|
| `courier_id` | int | |
| `city` | str | from `zones.csv` |
| `vehicle_type` | str | |
| `vehicle_age_years` | int | `2024 - purchase_year` |
| `days_since_service` | int | days since `last_service_date` |
| `age` | int | |
| `tenure_days` | int | days since `joined_date` |
| `rating` | float | |
| `total_earnings` | float | |
| `number_of_orders` | int | all orders assigned to the courier, any status |
| `number_of_five_star_orders` | int | delivered orders with `rating_given == 5` |
| `on_time_rate` | float | share of the courier's **delivered** orders with `on_time == True`, rounded to 4 decimals; use `0.0` if the courier has no delivered orders |
| `courier_tier` | str | |

Rows and columns may be in any order — the tests are order-agnostic.

In [184]:
# YOUR SOLUTION — Question 2
_t0 = time.time()

couriers = pd.read_json('q2/couriers.json')
vehicles = pd.read_csv('q2/vehicles.csv')
zones = pd.read_csv('q2/zones.csv')
orders = pd.concat([pd.read_csv('q2/orders_1.csv'), pd.read_csv('q2/orders_2.csv'), pd.read_csv('q2/orders_3.csv')])
TODAY = pd.to_datetime("2024-06-1")
# print(TODAY)
merged = couriers.merge(
    zones,
    on='zone_id'
)
print(len(couriers))

print(len(merged))
merged = merged.merge(
    vehicles,
    on='vehicle_id'
)

merged['vehicle_age_years'] = (
    2024- merged['purchase_year']
)

merged['days_since_service'] = (
    TODAY - pd.to_datetime(merged['last_service_date'], format='%d/%m/%Y')
).dt.days

merged['tenure_days'] = (
    TODAY - pd.to_datetime(merged['joined_date'])
).dt.days

print(len(merged))
# print(merged)
# print(orders)
# ratings = orders.groupby('courier_id')['rating_given'].dropna(columns='rating_given').mean().reset_index()
ratings = orders.dropna(subset='rating_given').groupby('courier_id')['rating_given'].mean().reset_index()
print(len(merged))
# earnings = orders.dropna(subset='total').groupby('courier_id')['rating_given'].mean().reset_index()
# print(ratings)
total_orders = orders.groupby('courier_id').size().rename("number_of_orders").reset_index()
print(len(merged))

number_of_five_star_orders = orders[
    orders['rating_given']==5
].groupby('courier_id').size().rename("number_of_five_star_orders").reset_index()
# print(number_of_five_star_orders)

print(len(merged))
# on_time_rate = orders.groupby('courier_id')['on_time'].sum().rename("on_time_num").reset_index()
on_time_rate = orders.groupby('courier_id')['on_time'].mean().round(4).rename("on_time_rate").reset_index()
print(on_time_rate)
merged = merged.merge(
    on_time_rate[['courier_id', 'on_time_rate']],
    on='courier_id'
)
# on_time_rate = on_time_rate.merge(
#     total_orders,
#     on='courier_id'
# )
# on_time_rate['on_time_rate'] = round(on_time_rate['on_time_num']/on_time_rate['number_of_orders'],4)
# print(on_time_rate)

# print(total_orders)
merged = merged.merge(
    ratings,
    on='courier_id'
)
merged = merged.merge(
    total_orders,
    on='courier_id'
)
merged = merged.merge(
    number_of_five_star_orders,
    on='courier_id',
    how='left'
)
merged['number_of_five_star_orders'] = merged['number_of_five_star_orders'].fillna(0).astype(int)


final = merged[['courier_id', 'city', 'vehicle_type', 'vehicle_age_years', 'days_since_service', 'age', 'tenure_days', 'rating', 'total_earnings', 'number_of_orders', 'number_of_five_star_orders', 'on_time_rate', 'courier_tier']]

final.to_csv('q2/collected.csv', index=False)
# ... write q2/collected.csv
# print(f"runtime: {time.time() - _t0:.2f}s")

2500
2500
2500
2500
2500
2500
      courier_id  on_time_rate
0           5000        0.6250
1           5001        0.7742
2           5002        0.5588
3           5003        0.5517
4           5004        0.3333
...          ...           ...
2495        7495        0.6154
2496        7496        0.6429
2497        7497        0.9310
2498        7498        0.7838
2499        7499        0.7500

[2500 rows x 2 columns]


In [185]:
grader.grade_q2()

=== Round 2 / Question 2: feature collection ===
[PASS] collected.csv exists  -> /Users/log/Github/CPT/capital1_oa/round2/q2/collected.csv
[PASS] file is tab-separated (>1 column parsed)  -> 13 column(s)
[PASS] all required columns present  -> missing []
[PASS] row count == 2500  -> got 2500
[PASS] courier_id set matches
[PASS] city matches  -> 0 mismatched rows
[PASS] vehicle_type matches  -> 0 mismatched rows
[PASS] vehicle_age_years matches  -> 0 mismatched rows
[PASS] days_since_service matches  -> 0 mismatched rows
[PASS] age matches  -> 0 mismatched rows
[PASS] tenure_days matches  -> 0 mismatched rows
[PASS] rating matches  -> 0 mismatched rows
[PASS] total_earnings matches  -> 0 mismatched rows
[PASS] number_of_orders matches  -> 0 mismatched rows
[PASS] number_of_five_star_orders matches  -> 0 mismatched rows
[FAIL] on_time_rate matches  -> 2471 mismatched rows
[PASS] courier_tier matches  -> 0 mismatched rows
--- 16/17 checks passed ---


False

---
## Question 3 of 4 — Data preparation

A dataset of couriers and their performance metrics, with columns:

`courier_id` (int), `city` (str), `vehicle_type` (str), `vehicle_age_years` (int),
`days_since_service` (int), `age` (int), `tenure_days` (int), `rating` (float), `total_earnings` (float),
`number_of_orders` (int), `number_of_five_star_orders` (int), `on_time_rate` (float),
`number_of_cancellations` (int), `number_of_complaints` (int), `avg_delivery_minutes` (float),
`courier_tier` (str)

Split: **train 75%** at `q3/data/train.csv`, **test 25%** at `q3/data/test.csv`.

**Steps**
- **a.** Fill missing values in `avg_delivery_minutes` with the **median**, rounded to 2 decimal places.
- **b.** Encode `city` with **ordinal encoding in alphabetical order**, starting at 0 and consecutive
  (`Austin` → 0, `Boston` → 1, …).
- **c.** Encode `vehicle_type` with **one-hot encoding**: add one column per type named
  `vehicle_type_<value>` (e.g. `vehicle_type_Car`) holding 0/1, and drop the original `vehicle_type`
  column. Both output files must have the same one-hot columns.
- **d.** Normalize `total_earnings` with **Min-Max scaling** to the range [0, 1].
- **e.** Convert `courier_tier`: `"Standard"` → 0, `"Gold"` → 1.

⚠️ **No leakage** — every statistic (median, category list, min/max) is fit on **train only**.

**Output:** `q3/processed_train.csv` and `q3/processed_test.csv`.
Values in `total_earnings` must be written with **exactly 4 digits after the decimal point**.

**Constraints:** 8 s, 4 GB.

In [ ]:
# YOUR SOLUTION — Question 3
_t0 = time.time()

train = pd.read_csv("q3/data/train.csv")
test = pd.read_csv("q3/data/test.csv")

train['avg_delivery_minutes'] = train['avg_delivery_minutes'].fillna(round(train['avg_delivery_minutes'].median(),2))
test['avg_delivery_minutes'] = test['avg_delivery_minutes'].fillna(round(train['avg_delivery_minutes'].median(),2))

# print(round(train['avg_delivery_minutes'].median(),2))
import sklearn.preprocessing
encoder = sklearn.preprocessing.OrdinalEncoder()
train[['city']] = encoder.fit_transform(
    train[['city']]
)
test[['city']] = encoder.transform(
    test[['city']]
)

print(train['vehicle_type'].unique())
types = ['Car', 'Bicycle', 'Walking', 'E-bike', 'Scooter']
for type in types:
    train[f'vehicle_type_{type}'] = (train['vehicle_type']==type).astype(int)
    test[f'vehicle_type_{type}'] = (test['vehicle_type']==type).astype(int)
train = train.drop(columns=["vehicle_type"])
test = test.drop(columns=["vehicle_type"])

scaler = sklearn.preprocessing.MinMaxScaler()
train[['total_earnings']] = scaler.fit_transform(
    train[['total_earnings']]
)
train[['total_earnings']] = train[['total_earnings']].map(
    lambda x: f"{x:.4f}"
)
test[['total_earnings']] = scaler.transform(
    test[['total_earnings']]
)
test[['total_earnings']] = test[['total_earnings']].map(
    lambda x: f"{x:.4f}"
)

train['courier_tier'] = train['courier_tier'].replace({
    "Standard": 0,
    "Gold": 1
})
test['courier_tier'] = test['courier_tier'].replace({
    "Standard": 0,
    "Gold": 1
})

# print(train)
train.to_csv('q3/processed_train.csv')
test.to_csv('q3/processed_test.csv')
# ... write q3/processed_train.csv and q3/processed_test.csv

print(f"runtime: {time.time() - _t0:.2f}s  (limit 8s)")

['Car' 'Bicycle' 'Walking' 'E-bike' 'Scooter']
runtime: 0.02s  (limit 8s)


/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_98563/2900862224.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['courier_tier'] = train['courier_tier'].replace({
/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_98563/2900862224.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test['courier_tier'] = test['courier_tier'].replace({


In [240]:
grader.grade_q3()

=== Round 2 / Question 3: preprocessing ===
[PASS] processed_train.csv exists
[PASS] processed_test.csv exists
[PASS] output is tab-separated
[PASS] train row count preserved  -> 1875 vs 1875
[PASS] test row count preserved  -> 625 vs 625
[PASS] avg_delivery_minutes has no missing values
[PASS] train gaps filled with TRAIN median rounded to 2dp (32.84)  -> got [32.84]
[PASS] test gaps filled with TRAIN median rounded to 2dp (32.84)  -> got [32.84]
[PASS] city is numeric
[PASS] city encoded alphabetically 0..4  -> expected {'Austin': 0, 'Boston': 1, 'Chicago': 2, 'Denver': 3, 'Seattle': 4}
[PASS] test city uses the same mapping
[PASS] one-hot columns present (5)  -> missing set()
[PASS] original vehicle_type column dropped
[PASS] one-hot values are 0/1
[PASS] exactly one 1 per row
[PASS] one-hot matches the source vehicle_type
[PASS] test has the same one-hot columns
[PASS] train total_earnings min-max scaled to [0,1]  -> min=0.0000 max=1.0000
[PASS] test scaled with TRAIN min/max (no l

True

---
## Question 4 of 4 — Classifier

Train a classifier that predicts whether a courier is **Gold tier (1)** or **Standard tier (0)**.

Free-form — any model, any libraries.

**Data (in `q4/`)**
- Training set 70% — `q4/data/train.csv`
- Validation set 15% — `q4/data/val.csv`
- Test set 15% — `q4/data/test.csv` (no `courier_tier` column)

**Metrics:** precision and recall with **Gold as the positive class**.
**Goal:** maximize **F1** — balance precision and recall rather than pushing one to the extreme.
Roughly 40% of couriers are Gold.

**Output:** `q4/predictions.csv` — one column named `courier_tier`, one row per test row, in test-set order:

```
courier_tier
0
1
0
...
```

The grader prints a first-10-row preview score, then the full precision / recall / F1.

**Constraints:** 8 s, 4 GB.

In [279]:
# YOUR SOLUTION — Question 4
_t0 = time.time()

train = pd.read_csv("q4/data/train.csv")
val = pd.read_csv("q4/data/val.csv")
test = pd.read_csv("q4/data/test.csv")


scaler = sklearn.preprocessing.MinMaxScaler()
train[['total_earnings']] = scaler.fit_transform(
    train[['total_earnings']]
)
val[['total_earnings']] = scaler.transform(
    val[['total_earnings']]
)
test[['total_earnings']] = scaler.transform(
    test[['total_earnings']]
)

import sklearn.preprocessing
encoder = sklearn.preprocessing.OrdinalEncoder()
train[['city']] = encoder.fit_transform(
    train[['city']]
)
val[['city']] = encoder.transform(
    val[['city']]
)
test[['city']] = encoder.transform(
    test[['city']]
)
train['avg_delivery_minutes'] = train['avg_delivery_minutes'].fillna(round(train['avg_delivery_minutes'].median(),2))
test['avg_delivery_minutes'] = test['avg_delivery_minutes'].fillna(round(train['avg_delivery_minutes'].median(),2))
val['avg_delivery_minutes'] = val['avg_delivery_minutes'].fillna(round(train['avg_delivery_minutes'].median(),2))

types = ['Car', 'Bicycle', 'Walking', 'E-bike', 'Scooter']
for type in types:
    train[f'vehicle_type_{type}'] = (train['vehicle_type']==type).astype(int)
    test[f'vehicle_type_{type}'] = (test['vehicle_type']==type).astype(int)
    val[f'vehicle_type_{type}'] = (val['vehicle_type']==type).astype(int)
train = train.drop(columns=["vehicle_type"])
test = test.drop(columns=["vehicle_type"])
val = val.drop(columns=["vehicle_type"])

train['courier_tier'] = train['courier_tier'].replace({
    "Standard": 0,
    "Gold": 1
})
val['courier_tier'] = val['courier_tier'].replace({
    "Standard": 0,
    "Gold": 1
})

import sklearn.linear_model

model = sklearn.linear_model.LogisticRegression()

X_train = train.drop(columns='courier_tier')
Y_train = train['courier_tier']

X_val = val.drop(columns='courier_tier')
Y_val = val['courier_tier']

# print(train.isna().sum())
# print(train)


model.fit(X_train, Y_train)
import sklearn.metrics

# preds = model.predict(X_val)
thresholds = [0.05, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
for threshold in thresholds:
    probs = model.predict_proba(X_val)
    preds = (probs >= threshold)[:, 1].astype(int)
    # print(preds)
    print(threshold)
    print(sklearn.metrics.f1_score(preds, Y_val))
    print(sklearn.metrics.recall_score(preds, Y_val))
    print(sklearn.metrics.precision_score(preds, Y_val))

test_probs = model.predict_proba(test)
test_preds = (probs >= 0.3)[:, 1].astype(int)
# print(test_preds)
test_pred_df = pd.Series({
    "courier_type": test_preds
})
test_pred_df.to_csv('q4/predictions.csv', index=False)
# ... write q4/predictions.csv

print(f"runtime: {time.time() - _t0:.2f}s  (limit 8s)")

0.05
0.7828282828282829
0.6431535269709544
1.0
0.2
0.8757396449704142
0.8087431693989071
0.9548387096774194
0.3
0.88125
0.8545454545454545
0.9096774193548387
0.4
0.8618421052631579
0.8791946308724832
0.8451612903225807
0.5
0.8552188552188552
0.8943661971830986
0.8193548387096774
0.6
0.8501742160278746
0.9242424242424242
0.7870967741935484
0.7
0.8345323741007195
0.943089430894309
0.7483870967741936
runtime: 0.21s  (limit 8s)


/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_98563/2423613087.py:44: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['courier_tier'] = train['courier_tier'].replace({
/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_98563/2423613087.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['courier_tier'] = val['courier_tier'].replace({
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (statu

In [280]:
grader.grade_q4()

=== Round 2 / Question 4: classifier ===
[PASS] predictions.csv exists
[FAIL] single column named 'courier_tier'  -> ['0']
--- 1/2 checks passed ---


False

---
### Run everything

In [ ]:
for fn in (grader.grade_q1, grader.grade_q2, grader.grade_q3, grader.grade_q4):
    fn()
    print()